In [1]:
import os
import traceback
import pandas as pd
from datetime import datetime
from pyswmm import Simulation, Nodes, Links

# File paths
input_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp"
hotstart_hsf_file = "hotstart_harris_no_event.hsf"
output_csv_file = "node_depths_daily_no_event.csv"

# Run SWMM simulation
def run_swmm_simulation(input_path, start_time, end_time, hotstart_hsf_file, output_csv_file):
    print(f"Running SWMM simulation from {start_time} to {end_time}...")

    try:
        restart_simulation = os.path.exists(hotstart_hsf_file)

        with Simulation(input_path) as sim:
            if restart_simulation:
                print("Hotstart file found. Applying initial conditions...")
                sim.use_hotstart(hotstart_hsf_file)
            else:
                print("No hotstart file found. Starting simulation from scratch...")

            sim.start_time = start_time
            sim.end_time = end_time
            sim.step_advance(900)  # 15-minute steps


            # Store node references for fast access
            nodes = {node.nodeid: node for node in Nodes(sim)}

            # Manually advance one step to apply hotstart
            print("\nAdvancing one step to apply hotstart...")
            next(sim)

            # Dictionary to store daily depth variations
            daily_depths = {}

            print("Starting full simulation...")
            for step in sim:
                current_time = sim.current_time
                date_str = str(current_time.date())

                # Log progress every 6 hours
                if current_time.hour % 6 == 0 and current_time.minute == 0:
                    print(f"Simulation Time: {current_time}")

                # Store depth variations throughout the day
                for node_id, node in nodes.items():
                    if (date_str, node_id) not in daily_depths:
                        daily_depths[(date_str, node_id)] = []
                    daily_depths[(date_str, node_id)].append(node.depth)  # Collect depth changes

            # Process depths: Calculate min, max, and average per day
            node_depths = []
            for (date, node_id), depth_list in daily_depths.items():
                node_depths.append([date, node_id, min(depth_list), max(depth_list), sum(depth_list) / len(depth_list)])

            # Debug print before saving
            print("\nChecking recorded node depths before saving to CSV:")
            if node_depths:
                for entry in node_depths[:5]:  # Print first 5 entries
                    print(entry)
            else:
                print("No depths recorded! Check simulation step timing.")

            # Convert recorded depths to DataFrame & Save as CSV
            df_depths = pd.DataFrame(node_depths, columns=["Date", "Node ID", "Min Depth", "Max Depth", "Avg Depth"])
            df_depths.to_csv(output_csv_file, index=False)
            print(f"Daily node depths saved to {output_csv_file}")

    except Exception as e:
        print("Error during simulation:", e)
        traceback.print_exc()

# Main workflow
if __name__ == "__main__":
    print("Starting optimized SWMM workflow...")

    # Define simulation period
    full_start_time = datetime(2021, 6, 15, 0, 0, 0)  # Start date
    full_end_time = datetime(2021, 6, 28, 0, 0, 0)    # End date

    # Run simulation on modified .inp file
    run_swmm_simulation(
        input_path,
        full_start_time,
        full_end_time,
        hotstart_hsf_file,
        output_csv_file,
    )


Starting optimized SWMM workflow...
Running SWMM simulation from 2021-06-15 00:00:00 to 2021-06-28 00:00:00...
Hotstart file found. Applying initial conditions...

Advancing one step to apply hotstart...
Starting full simulation...
Simulation Time: 2021-06-15 06:00:00
Simulation Time: 2021-06-15 12:00:00
Simulation Time: 2021-06-15 18:00:00
Simulation Time: 2021-06-16 00:00:00
Simulation Time: 2021-06-16 06:00:00
Simulation Time: 2021-06-16 12:00:00
Simulation Time: 2021-06-16 18:00:00
Simulation Time: 2021-06-17 00:00:00
Simulation Time: 2021-06-17 06:00:00
Simulation Time: 2021-06-17 12:00:00
Simulation Time: 2021-06-17 18:00:00
Simulation Time: 2021-06-18 00:00:00
Simulation Time: 2021-06-18 06:00:00
Simulation Time: 2021-06-18 12:00:00
Simulation Time: 2021-06-18 18:00:00
Simulation Time: 2021-06-19 00:00:00
Simulation Time: 2021-06-19 06:00:00
Simulation Time: 2021-06-19 12:00:00
Simulation Time: 2021-06-19 18:00:00
Simulation Time: 2021-06-20 00:00:00
Simulation Time: 2021-06-20 

Node JCT-458: Updated Depth = 0.02009986899793148
Node JCT-46: Updated Depth = 0.09661929309368134
Node JCT-462: Updated Depth = 0.3782305121421814
Node JCT-464: Updated Depth = 0.17714166641235352
Node JCT-466: Updated Depth = 0.2301744818687439
Node JCT-468: Updated Depth = 0.24359585344791412
Node JCT-470: Updated Depth = 0.28038766980171204
Node JCT-472: Updated Depth = 0.24355439841747284
Node JCT-478: Updated Depth = 0.44209253787994385
Node JCT-48: Updated Depth = 0.19207151234149933
Node JCT-480: Updated Depth = 0.1684640645980835
Node JCT-482: Updated Depth = 0.2469625473022461
Node JCT-484: Updated Depth = 0.2461380809545517
Node JCT-486: Updated Depth = 0.16523128747940063
Node JCT-488: Updated Depth = 0.1649000644683838
Node JCT-492: Updated Depth = 0.21693527698516846
Node JCT-496: Updated Depth = 0.210718035697937
Node JCT-50: Updated Depth = 0.20966117084026337
Node JCT-500: Updated Depth = 0.13254933059215546
Node JCT-502: Updated Depth = 0.20829090476036072
Node JCT-50

Node RVSDS_INT_RVI_XIV_MH14-06: Updated Depth = 1.3343720436096191
Node RVSDS_INT_RVI_XIV_MH14-07: Updated Depth = 1.3348369598388672
Node RVSDS_INT_RVI_XIV_MH14-08: Updated Depth = 1.3328801393508911
Node RVSDS_INT_RVI_XIV_MH14-09: Updated Depth = 1.344067931175232
Node RVSDS_INT_RVI_XIV_MH14-10: Updated Depth = 1.32096529006958
Node RVSDS_INT_RVI_XIV_MH14-11: Updated Depth = 1.290115475654602
Node RVSDS_INT_RVI_XIV_MH14-12: Updated Depth = 1.2276310920715332
Node RVSDS_INT_RVI_XIV_MH14-13: Updated Depth = 1.1868449449539185
Node RVSDS_INT_RVI_XIV_MH14-14: Updated Depth = 1.177374243736267
Node RVSDS_INT_RVI_XIV_MH14-15: Updated Depth = 1.1864858865737915
Node RVSDS_INT_RVI_XIV_MH14-16: Updated Depth = 1.1775789260864258
Node RVSDS_INT_RVI_XIV_MH14-17: Updated Depth = 1.180776596069336
Node RVSDS_INT_RVI_XIV_MH14-18: Updated Depth = 1.1847130060195923
Node RVSDS_INT_RVI_XIV_MH14-19: Updated Depth = 1.1555957794189453
Node RVSDS_INT_RVI_XIV_MH14-20: Updated Depth = 1.1369963884353638
N

Node SFIT2874: Updated Depth = 0.1934019923210144
Node SFIT2877: Updated Depth = 0.13997577130794525
Node SFIT2878: Updated Depth = 0.307900071144104
Node SFIT2879: Updated Depth = 0.22021816670894623
Node SFIT2880: Updated Depth = 0.12842747569084167
Node SFIT2882: Updated Depth = 0.0526278056204319
Node SFIT2883: Updated Depth = 0.021411918103694916
Node SFIT2901: Updated Depth = 0.32584890723228455
Node SFIT2905: Updated Depth = 0.22163327038288116
Node SFIT2906: Updated Depth = 0.19558602571487427
Node SFIT2907: Updated Depth = 0.2034740000963211
Node SFIT2910: Updated Depth = 0.37472379207611084
Node SFIT2912: Updated Depth = 0.15974698960781097
Node SFIT2915: Updated Depth = 0.3383861780166626
Node SFIT2916: Updated Depth = 0.38712090253829956
Node SFIT2918: Updated Depth = 0.3274247348308563
Node SFIT2919: Updated Depth = 0.37296563386917114
Node SFIT2922: Updated Depth = 0.2036447525024414
Node SFIT2923: Updated Depth = 0.2752857506275177
Node SFIT2925: Updated Depth = 0.260183

Node SMH14087: Updated Depth = 0.15741536021232605
Node SMH14090: Updated Depth = 0.5269875526428223
Node SMH14100: Updated Depth = 0.2661344110965729
Node SMH14103: Updated Depth = 0.3733932375907898
Node SMH14116: Updated Depth = 0.21088190376758575
Node SMH14120: Updated Depth = 0.03747987002134323
Node SMH14121: Updated Depth = 0.10112228989601135
Node SMH14122: Updated Depth = 0.19812411069869995
Node SMH14129: Updated Depth = 0.20237421989440918
Node SMH14130: Updated Depth = 0.13727162778377533
Node SMH14139: Updated Depth = 0.09092634171247482
Node SMH14142: Updated Depth = 0.10857841372489929
Node SMH14143: Updated Depth = 0.12093841284513474
Node SMH14151: Updated Depth = 0.7787419557571411
Node SMH14153: Updated Depth = 0.11385846138000488
Node SMH14175: Updated Depth = 0.1000993624329567
Node SMH14177: Updated Depth = 0.1948087215423584
Node SMH14181: Updated Depth = 0.09214019775390625
Node SMH14185: Updated Depth = 0.13330595195293427
Node SMH14186: Updated Depth = 0.0
No

Node SMH34526: Updated Depth = 0.34977951645851135
Node SMH34534: Updated Depth = 0.23129132390022278
Node SMH34540: Updated Depth = 9.99339172267355e-05
Node SMH34541: Updated Depth = 9.998031600844115e-05
Node SMH34545: Updated Depth = 0.3088071644306183
Node SMH34549: Updated Depth = 9.999380563385785e-05
Node SMH34556: Updated Depth = 0.15690918266773224
Node SMH34558: Updated Depth = 0.0
Node SMH34561: Updated Depth = 0.00021174094581510872
Node SMH34564: Updated Depth = 0.46915340423583984
Node SMH34577: Updated Depth = 0.18413744866847992
Node SMH34579: Updated Depth = 0.17433065176010132
Node SMH34591: Updated Depth = 0.5731741786003113
Node SMH34621: Updated Depth = 0.48526570200920105
Node SMH34631: Updated Depth = 0.09373688697814941
Node SMH34655: Updated Depth = 0.45133310556411743
Node SMH34681: Updated Depth = 0.1214732825756073
Node SMH34690: Updated Depth = 0.270695298910141
Node SMH34692: Updated Depth = 0.09312404692173004
Node SMH34693: Updated Depth = 0.35632854700

Node SMH54891: Updated Depth = 0.25289419293403625
Node SMH54892: Updated Depth = 0.16677036881446838
Node SMH54895: Updated Depth = 0.22856514155864716
Node SMH54897: Updated Depth = 0.06400999426841736
Node SMH54898: Updated Depth = 0.1689402014017105
Node SMH54899: Updated Depth = 0.12923204898834229
Node SMH54900: Updated Depth = 0.2045217901468277
Node SMH54901: Updated Depth = 0.20737521350383759
Node SMH54902: Updated Depth = 0.21303875744342804
Node SMH54903: Updated Depth = 0.16566967964172363
Node SMH54905: Updated Depth = 0.1537402868270874
Node SMH54906: Updated Depth = 0.13109707832336426
Node SMH54908: Updated Depth = 0.13845165073871613
Node SMH54909: Updated Depth = 0.036892395466566086
Node SMH54911: Updated Depth = 0.3388560116291046
Node SMH54912: Updated Depth = 0.5096037983894348
Node SMH54917: Updated Depth = 1.7251057624816895
Node SMH54917b: Updated Depth = 9.933282126439735e-05
Node SMH55204: Updated Depth = 8.794612884521484
Node SMH60685: Updated Depth = 3.02

Node SFIT2769: Updated Depth = 0.268789142370224
Node SFIT2839: Updated Depth = 1.6679505109786987
Node SFIT3022: Updated Depth = 0.2864020764827728
Node SFIT3070: Updated Depth = 4.637382984161377
Node SFIT3092: Updated Depth = 0.11965635418891907
Node SFIT3093: Updated Depth = 0.11965728551149368
Node SFIT3101: Updated Depth = 0.01601971872150898
Node SFIT3104: Updated Depth = 0.14381171762943268
Node SFIT3118: Updated Depth = 0.28396153450012207
Node SFIT3268: Updated Depth = 0.1769018918275833
Node SFIT3384: Updated Depth = 0.03654831647872925
Node SFIT3392: Updated Depth = 0.21724484860897064
Node SFIT3424: Updated Depth = 0.3195894956588745
Node SFIT3443: Updated Depth = 0.1305701732635498
Node SFIT3482: Updated Depth = 7.188457012176514
Node SFIT3483: Updated Depth = 7.218457221984863
Node SFIT3543: Updated Depth = 0.2480044811964035
Node SFIT3649: Updated Depth = 0.24498215317726135
Node SFIT3981: Updated Depth = 0.11930285394191742
Node SFIT4194: Updated Depth = 9.980197501135

In [1]:
import os
import traceback
import pandas as pd
import numpy as np
from datetime import datetime
from pyswmm import Simulation, Nodes

# File paths
input_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event_3.inp"
hotstart_hsf_file = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\hotstart_harris_no_event.hsf"
output_csv_file = "node_depths_optimized.csv"

# **2️⃣ Run SWMM Simulation Using PySWMM**
def run_swmm_simulation(input_path, start_time, end_time, hotstart_hsf_file, output_csv_file):
    print(f"Running SWMM simulation from {start_time} to {end_time}...")

    try:
        restart_simulation = os.path.exists(hotstart_hsf_file)
        with Simulation(input_path) as sim:
            if restart_simulation:
                print("Hotstart file found. Applying initial conditions...")
                sim.use_hotstart(hotstart_hsf_file)
            else:
                print("No hotstart file found. Starting simulation from scratch...")

            # **Set time and step advance**
            sim.start_time = start_time
            sim.end_time = end_time  

            # **Store node references**
            nodes = {node.nodeid: node for node in Nodes(sim)}

            # Manually advance one step to apply hotstart
            print("\nAdvancing one step to apply hotstart...")
            next(sim)

            # **Tracking max/min depths using NumPy arrays**
            max_depths = {node_id: 0.0 for node_id in nodes}
            total_depths = {node_id: 0.0 for node_id in nodes}
            count_depths = {node_id: 0 for node_id in nodes}

            print("Starting full simulation...")
            for step in sim:
                current_time = sim.current_time

                # **Log progress every 24 hours**
                if current_time.hour % 24 == 0 and current_time.minute == 0:
                    print(f"Simulation Time: {current_time}")

                # **Efficiently compute min, max, and avg depths**
                for node_id, node in nodes.items():
                    depth = node.depth
                    max_depths[node_id] = max(max_depths[node_id], depth)
                    total_depths[node_id] += depth
                    count_depths[node_id] += 1

            # **Convert results to DataFrame & Save**
            df_depths = pd.DataFrame({
                "Node ID": list(nodes.keys()),
                "Max Depth": [max_depths[node_id] for node_id in nodes],
            })
            df_depths.to_csv(output_csv_file, index=False)
            print(f"Node depths saved to {output_csv_file}")

    except Exception as e:
        print("Error during simulation:", e)
        traceback.print_exc()

# **3️⃣ Main Execution**
if __name__ == "__main__":
    print("Starting optimized PySWMM workflow...")

    # Define simulation period
    full_start_time = datetime(2021, 6, 15, 0, 0, 0)
    full_end_time = datetime(2021, 6, 28, 0, 0, 0)

    run_swmm_simulation(
        input_path,
        full_start_time,
        full_end_time,
        hotstart_hsf_file,
        output_csv_file,
    )


Starting optimized PySWMM workflow...
Running SWMM simulation from 2021-06-15 00:00:00 to 2021-06-28 00:00:00...
Hotstart file found. Applying initial conditions...

Advancing one step to apply hotstart...
Starting full simulation...
Simulation Time: 2021-06-15 00:00:06
Simulation Time: 2021-06-15 00:00:14
Simulation Time: 2021-06-15 00:00:22
Simulation Time: 2021-06-15 00:00:27
Simulation Time: 2021-06-15 00:00:35
Simulation Time: 2021-06-15 00:00:40
Simulation Time: 2021-06-15 00:00:44
Simulation Time: 2021-06-15 00:00:52
Simulation Time: 2021-06-15 00:00:59
Simulation Time: 2021-06-16 00:00:00
Simulation Time: 2021-06-16 00:00:08
Simulation Time: 2021-06-16 00:00:12
Simulation Time: 2021-06-16 00:00:15
Simulation Time: 2021-06-16 00:00:22
Simulation Time: 2021-06-16 00:00:29
Simulation Time: 2021-06-16 00:00:37
Simulation Time: 2021-06-16 00:00:41
Simulation Time: 2021-06-16 00:00:45
Simulation Time: 2021-06-16 00:00:51
Simulation Time: 2021-06-16 00:00:58
Simulation Time: 2021-06-1

In [ ]:
import os
import traceback
import pandas as pd
from datetime import datetime
from pyswmm import Simulation, Nodes

# File paths
input_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp"
hotstart_hsf_file = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\hotstart_harris_no_event.hsf"
output_csv_file = "node_depths_statistics_based.csv"

def run_swmm_simulation(input_path, start_time, end_time, hotstart_hsf_file, output_csv_file):
    print(f"Running SWMM simulation from {start_time} to {end_time}...")

    try:
        use_hotstart = os.path.exists(hotstart_hsf_file)

        with Simulation(input_path) as sim:
            if use_hotstart:
                print("Hotstart file found. Applying initial conditions...")
                sim.use_hotstart(hotstart_hsf_file)
            else:
                print("No hotstart file found. Starting simulation from scratch...")

            # Set simulation period and step
            sim.start_time = start_time
            sim.end_time = end_time

            nodes = Nodes(sim)

             # Prime hotstart file (optional first step)
            print("\nAdvancing one step to apply hotstart...")
            next(sim)

            print("Starting simulation...")
            for step in enumerate(sim):
                pass

            print("Simulation complete. Extracting statistics...")

            # Collect results using node statistics
            data = []
            for node in nodes:
                max_depth = node.statistics["max_depth"]

                data.append({
                    "Node ID": node.nodeid,
                    "Max Depth (ft)": max_depth
                })

            df = pd.DataFrame(data)
            df.to_csv(output_csv_file, index=False)
            print(f"\nResults saved to: {output_csv_file}")

    except Exception as e:
        print("❌ Error during simulation:", e)
        traceback.print_exc()

# === Main Execution ===
if __name__ == "__main__":
    print("🚀 Starting SWMM simulation using node.statistics...")

    start_time = datetime(2021, 6, 15, 0, 0, 0)
    end_time = datetime(2021, 6, 28, 0, 0, 0)

    run_swmm_simulation(
        input_path,
        start_time,
        end_time,
        hotstart_hsf_file,
        output_csv_file,
    )


🚀 Starting SWMM simulation using node.statistics...
Running SWMM simulation from 2021-06-15 00:00:00 to 2021-06-28 00:00:00...
Hotstart file found. Applying initial conditions...

Advancing one step to apply hotstart...
Starting simulation...


In [6]:
import os
import traceback
import pandas as pd
from datetime import datetime
from pyswmm import Simulation, Nodes

# === File paths ===
input_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event.inp"
hotstart_hsf_file = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\hotstart_harris_no_event.hsf"
output_csv_file = "node_depths_statistics_based_new2.csv"

# === Run SWMM Simulation Using PySWMM ===
def run_swmm_simulation(input_path, warmup_start, analysis_start, end_time, hotstart_hsf_file, output_csv_file):
    print(f"Running SWMM simulation from {warmup_start} to {end_time}...")

    try:
        use_hotstart = os.path.exists(hotstart_hsf_file)

        with Simulation(input_path) as sim:
            if use_hotstart:
                print("Hotstart file found. Applying initial conditions...")
                sim.use_hotstart(hotstart_hsf_file)
            else:
                print("No hotstart file found. Starting simulation from scratch...")

            # Set simulation window and match reporting step
            sim.start_time = warmup_start
            sim.end_time = end_time

            nodes = Nodes(sim)

            print("\nAdvancing one step to apply hotstart...")
            next(sim)

            print("Starting simulation loop...")
            for step in sim:
                current_time = sim.current_time

                if current_time.hour % 24 == 0 and current_time.minute == 0:
                    print(f"Simulation Time: {current_time}")

                # No need to collect anything during simulation; we use node.statistics after

            print("Simulation complete. Extracting max_report_depth...")

            # Collect and cap reported depths
            data = []
            for node in nodes:
                max_depth = node.statistics["max_depth"]
                data.append({
                    "Node ID": node.nodeid,
                    "Max Depth (ft)": max_depth
                })

            df = pd.DataFrame(data)
            df.to_csv(output_csv_file, index=False)
            print(f"\nResults saved to: {output_csv_file}")

    except Exception as e:
        print("❌ Error during simulation:", e)
        traceback.print_exc()

# === Main Execution ===
if __name__ == "__main__":
    print("🚀 Starting PySWMM simulation aligned with SWMM GUI...")

    warmup_start = datetime(2021, 6, 1, 0, 0, 0)      # Warm-up starts
    analysis_start = datetime(2021, 6, 15, 0, 0, 0)    # Analysis window starts
    end_time = datetime(2021, 6, 28, 0, 0, 0)          # Simulation end

    run_swmm_simulation(
        input_path,
        warmup_start,
        analysis_start,
        end_time,
        hotstart_hsf_file,
        output_csv_file,
    )


🚀 Starting PySWMM simulation aligned with SWMM GUI...
Running SWMM simulation from 2021-06-01 00:00:00 to 2021-06-28 00:00:00...
Hotstart file found. Applying initial conditions...

Advancing one step to apply hotstart...
Starting simulation loop...
Simulation Time: 2021-06-01 00:00:06
Simulation Time: 2021-06-01 00:00:14
Simulation Time: 2021-06-01 00:00:22
Simulation Time: 2021-06-01 00:00:27
Simulation Time: 2021-06-01 00:00:35
Simulation Time: 2021-06-01 00:00:40
Simulation Time: 2021-06-01 00:00:44
Simulation Time: 2021-06-01 00:00:52
Simulation Time: 2021-06-01 00:00:59
Simulation Time: 2021-06-02 00:00:00
Simulation Time: 2021-06-02 00:00:08
Simulation Time: 2021-06-02 00:00:15
Simulation Time: 2021-06-02 00:00:23
Simulation Time: 2021-06-02 00:00:31
Simulation Time: 2021-06-02 00:00:39
Simulation Time: 2021-06-02 00:00:46
Simulation Time: 2021-06-02 00:00:54
Simulation Time: 2021-06-03 00:00:00
Simulation Time: 2021-06-03 00:00:04
Simulation Time: 2021-06-03 00:00:09
Simulation

In [2]:
import os
import subprocess
import traceback
import pandas as pd
import swmmio

# File paths
input_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_modified_pumps_on.inp"
swmm_exe = r"C:\Program Files\EPA SWMM 5.2.4 (64-bit)\runswmm.exe"
output_csv_file = "node_true_max_depths_from_gui_pumps_on.csv"

# Derived paths
rpt_path = input_path.replace(".inp", ".rpt")
out_path = input_path.replace(".inp", ".out")

def run_swmm_via_subprocess(input_path, rpt_path, out_path, swmm_exe):
    print(f"▶ Running SWMM via subprocess with {swmm_exe}")
    try:
        result = subprocess.run(
            [swmm_exe, input_path, rpt_path, out_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True
        )
        print("✔ SWMM run completed.")
        print("STDOUT:", result.stdout)
        print("STDERR:", result.stderr)

    except subprocess.CalledProcessError as e:
        print("❌ SWMM run failed:")
        print("STDOUT:", e.stdout)
        print("STDERR:", e.stderr)
        raise

def extract_max_depths_from_rpt(input_path, output_csv_file):
    print(f"▶ Extracting node max depths from model: {input_path}")
    try:
        model = swmmio.Model(input_path)  # ✅ pass .inp, not .rpt
        df = model.rpt.node_depth_summary
        df = df[["MaxNodeDepth"]]
        df.index.name = "Node ID"
        df.reset_index(inplace=True)
        df.rename(columns={"MaxNodeDepth": "Max Depth (ft)"}, inplace=True)
        df.to_csv(output_csv_file, index=False)
        print(f"✔ Exported max depths to {output_csv_file}")
    except Exception as e:
        print("❌ Error extracting data:", e)
        traceback.print_exc()

# === Main Execution ===
if __name__ == "__main__":
    print("🚀 Running SWMM using subprocess + swmmio...")

    # 1️⃣ Run the simulation with SWMM CLI engine
    run_swmm_via_subprocess(input_path, rpt_path, out_path, swmm_exe)

    # 2️⃣ Extract results from the model
    extract_max_depths_from_rpt(input_path, output_csv_file)


🚀 Running SWMM using subprocess + swmmio...
▶ Running SWMM via subprocess with C:\Program Files\EPA SWMM 5.2.4 (64-bit)\runswmm.exe
❌ SWMM run failed:
STDOUT: 
 o  Retrieving project data
 o  Simulating day: 0     hour: 4 
STDERR: 


CalledProcessError: Command '['C:\\Program Files\\EPA SWMM 5.2.4 (64-bit)\\runswmm.exe', 'C:\\Users\\csofios\\Desktop\\LongTermCSO\\SWMM\\GLWA_Core_v4_R05_2020_p2_harris_modified_pumps_on.inp', 'C:\\Users\\csofios\\Desktop\\LongTermCSO\\SWMM\\GLWA_Core_v4_R05_2020_p2_harris_modified_pumps_on.rpt', 'C:\\Users\\csofios\\Desktop\\LongTermCSO\\SWMM\\GLWA_Core_v4_R05_2020_p2_harris_modified_pumps_on.out']' returned non-zero exit status 1.

In [1]:
import traceback
import pandas as pd
import swmmio

# Path to the input .inp file (used by swmmio to locate the .rpt)
input_path = r"C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event_controls.inp"
output_csv_file = "node_true_max_depths_from_gui.csv"

def extract_max_depths_from_rpt(input_path, output_csv_file):
    print(f"▶ Extracting node max depths from model: {input_path}")
    try:
        model = swmmio.Model(input_path)  # This automatically reads the .rpt
        df = model.rpt.node_depth_summary
        df = df[["MaxNodeDepth"]]
        df.index.name = "Node ID"
        df.reset_index(inplace=True)
        df.rename(columns={"MaxNodeDepth": "Max Depth (ft)"}, inplace=True)
        df.to_csv(output_csv_file, index=False)
        print(f"✔ Exported max depths to {output_csv_file}")
    except Exception as e:
        print("❌ Error extracting data:", e)
        traceback.print_exc()

# ✅ Run only the extraction
extract_max_depths_from_rpt(input_path, output_csv_file)


▶ Extracting node max depths from model: C:\Users\csofios\Desktop\LongTermCSO\SWMM\GLWA_Core_v4_R05_2020_p2_harris_no_event_controls.inp
✔ Exported max depths to node_true_max_depths_from_gui.csv


C:\Users\csofios\AppData\Local\Temp\3\ipykernel_16060\3998668730.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"MaxNodeDepth": "Max Depth (ft)"}, inplace=True)
